<a href="https://colab.research.google.com/github/sumeet-darekar/Sekiato/blob/php-testing/php_vulnerable_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/sumeet-darekar/PHP-vulnerabilities-dataset.git

Cloning into 'PHP-vulnerabilities-dataset'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 36 (delta 14), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 699.50 KiB | 1.24 MiB/s, done.
Resolving deltas: 100% (14/14), done.


In [3]:
!pip install datasets

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 16.3 MB/s eta 0:00:00


In [4]:
import pandas as pd
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from datasets import Dataset

# Load the datasets
safe_data = pd.read_csv('/content/PHP-vulnerabilities-dataset/URF/CWE_601_safe.csv')
unsafe_data = pd.read_csv('/content/PHP-vulnerabilities-dataset/URF/CWE_601_unsafe.csv')


In [6]:
safe_data['label'] = 0
unsafe_data['label'] = 1

# Combine the datasets
data = pd.concat([safe_data, unsafe_data], ignore_index=True)

# Assuming the CSV has a column 'code' with the code snippets
data = data[['PHP_Code', 'label']]

# Split into train and test sets
train_df, test_df = train_test_split(data, test_size=0.2)

# Convert to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Initialize the tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained('microsoft/codebert-base')
model = RobertaForSequenceClassification.from_pretrained('microsoft/codebert-base', num_labels=2)



/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
def tokenize_function(examples):
    return tokenizer(examples['PHP_Code'], padding="max_length", truncation=True)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Define the training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,  # Adjust epochs as needed
    save_total_limit=2,
    load_best_model_at_end=True,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

# Train the model
trainer.train()

# Save the fine-tuned model
model.save_pretrained('./fine_tuned_codebert')



Map:   0%|          | 0/3840 [00:00<?, ? examples/s]

Map:   0%|          | 0/960 [00:00<?, ? examples/s]

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,No log,0.000018
2,0.029000,0.000009
3,0.000000,0.000007


In [22]:
def predict_code_snippet(code_snippet):
    # Load the fine-tuned model
    model = RobertaForSequenceClassification.from_pretrained('./fine_tuned_codebert')
    model.eval()

    # Tokenize the input code snippet
    inputs = tokenizer(code_snippet, return_tensors='pt', padding=True, truncation=True, max_length=512)

    # Perform prediction
    with torch.no_grad():
        outputs = model(**inputs)

    # Get the prediction (0 for safe, 1 for unsafe)
    prediction = torch.argmax(outputs.logits, dim=-1).item()

    return "Vulnerable" if prediction == 1 else "Safe"

# Example prediction
example_code = """

<?php $handle = @fopen("/tmp/tainted.txt", "r"); if ($handle) { if(($tainted = fgets($handle, 4096)) == false) { $tainted = ""; } fclose($handle); } else { $tainted = ""; } $re = "/^[a-zA-Z]*$/"; if(preg_match($re, $tainted) == 1){ $tainted = $tainted; } else{ $tainted = ""; } //flaw $var = http_redirect(sprintf("'%s'", $tainted)); ?>

"""
result = predict_code_snippet(example_code)
print(f"The code is: {result}")


The code is: Vulnerable
